# Site-token baseline: train, tweak, and inspect attention

Interactive companion to `run_four_model_comparison.py`. Choose which decoders to train, for how long, and with which hyperparameters, then compare their test scores and look at what each one attends to.

| model | what it is |
|---|---|
| `ridge_site` | closed-form ridge, one LOO alpha per (time bin, site); the reference, keep it first |
| `ridge_timebin` | one linear map per bin for all sites, fitted by SGD (slow; off by default) |
| `baseline` | `BaselineModel`: 10 learned time queries attend over the 3 x 1024 DINOv3 coordinates; one 32 -> N readout per bin |
| `site_token` | `SiteTokenBaselineModel`: query $q_{t,n} = e_t + s_n$, so every (time block, site) chooses its own features; one shared 32 -> 1 readout per bin |
| `site_token_single` | same model trained on (image, site) pairs, one site per pass |
| `site_token_matched` | joint `site_token` with batch / epochs chosen to match `site_token_single`'s update count |

**Feature attention.** For query $q$ and coordinate $(l, e)$, with $\tilde x$ the per-layer LayerNorm of the pooled feature,

$$a_{q,l,e} = \operatorname{softmax}_{(l,e)}\!\left(\frac{(q \cdot k_{l,e})\,\tilde x_{l,e}}{\sqrt{d_h}}\right), \qquad z_q = W_v\,(3072\; a_q \odot \tilde x).$$

The softmax runs over all $3 \times 1024 = 3072$ coordinates, separately per head, and the maps below average the heads. Attention is shown as **multiples of uniform** ($3072\,a$), so 1 means "this coordinate gets its fair share".

**How to use.** Edit the configuration cell and run all. Each trained model is cached under `output_root/<dataset>/<run_tag>/` together with the settings it was trained with. A rerun reloads a model whose settings did not change and retrains only the ones you edited (or all of them with `retrain=True`).

In [ ]:
import json
import math
import os
import sys
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

# Locate the repository whether Jupyter starts in the project root or scripts folder.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [cwd, *cwd.parents] if (path / "config.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate config.yaml from the current directory.")
# end if PROJECT_ROOT is None

ENV = os.getenv("MY_ENV", "tiziano_mac_mini")
with open(PROJECT_ROOT / "config.yaml", "r") as f:
    config = yaml.safe_load(f)
# end with project configuration
paths = config[ENV]["paths"]

# The run script owns the loaders, fitting functions, and metrics; reuse them.
for source_path in (
    paths["useful_stuff_path"],
    str(PROJECT_ROOT / "python_scripts" / "src"),
    str(PROJECT_ROOT / "python_scripts" / "scripts"),
):
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
    # end if source path is not registered
# end for source path

import importlib
import run_four_model_comparison as comparison
comparison = importlib.reload(comparison)
# The run script selects the Agg backend for batch runs; restore inline plots.
%matplotlib inline
from IT_recap.dynamic_drsa import prepared_rdm_time_series
from IT_recap.neural_prediction_training import (
    aggregate_attention_by_layer,
    stimulus_correlation,
)
from model_classes.temporal_models import BaselineModel, SiteTokenBaselineModel

## Configuration

`decoder` holds the settings shared by every model (the script's `Cfg`; every field there can be overridden). `model_overrides` changes them for one model only, e.g. a smaller batch for `baseline`, which otherwise stalls on three0 at the predict-the-mean plateau (only 8 updates per epoch with batch 64).

In [ ]:
@dataclass
class NotebookCfg:
    dataset: str = "three0"  # "three0" or "paul"
    seed: int = 0  # image split (60/20/20) and initialization seed
    # Trained in this order; ridge_site is the reference for the per-site plots.
    models: tuple = ("ridge_site", "baseline", "site_token")
    retrain: bool = False  # True ignores cached runs with matching settings
    run_tag: str = "default"  # cache folder: change it to keep variants apart
    output_root: Path = Path(paths["data_path"]) / "results" / "site_token_notebook"
    device: str = "auto"
    # Window for the per-site summaries (ms).
    summary_window_ms: tuple = (50.0, 300.0)
    # Images per forward pass when collecting attention maps.
    attention_batch_size: int = 16
    # Coordinates shown in the feature-level attention maps.
    n_top_features: int = 40
# EOC


nb_cfg = NotebookCfg()

# Shared decoder settings; see comparison.Cfg for every available field.
decoder = replace(
    comparison.Cfg(),
    epochs=200,
    batch_size=64,
    baseline_learning_rate=1e-3,
    baseline_weight_decay=1e-4,
    temporal_embedding_dim=256,
    value_dim=128,
    n_attention_heads=2,
    temporal_compression_ratio=3,  # fine bins per coarse query; must divide 30
    mlp_hidden_dim=32,
    dropout=0.3,
    attention_granularity="feature",  # "feature" or "layer"
    layer_names="layer.3.mlp.down_proj,layer.13.mlp.down_proj,layer.20.mlp.down_proj",
    print_every=20,  # validation printout every k epochs (0 = silent)
)

# Per-model changes on top of `decoder`.
model_overrides = {
    # "baseline": dict(batch_size=16),
    "site_token_matched": dict(batch_size=3, epochs=234),
}

## Data and split

Targets are the trial-averaged AIT natraster, 0-300 ms in 10 ms bins, centered on the fit images per (time, site). Inputs are the cached, mean-pooled DINOv3 ViT-L layers. The split is the same `make_image_splits` split the script uses, so a given seed matches `run_four_model_comparison.py`.

In [ ]:
device = comparison.resolve_device(nb_cfg.device)
targets, ann_index, features = comparison.load_dataset(paths, nb_cfg.dataset, decoder)
splits, centered, design, to_tensors = comparison.prepare_split(
    decoder, nb_cfg.seed, targets, ann_index, features, device
)
raw_data = to_tensors(features)  # attention models: raw layer vectors
shapes = (*features.shape[1:], *targets.shape[1:])  # (layers, embedding, time, sites)
n_layers, feature_dim, n_time, n_sites = shapes
time_ms = decoder.time_start_ms + np.arange(n_time) * 1000.0 / decoder.target_fs
layer_labels = [name.split(".")[1] for name in decoder.layer_names.split(",")]
test_targets = centered[splits["test"]]

run_dir = nb_cfg.output_root / nb_cfg.dataset / nb_cfg.run_tag
run_dir.mkdir(parents=True, exist_ok=True)
print(f"{nb_cfg.dataset} | device {device} | targets {targets.shape} | features {features.shape}")
print({split: len(rows) for split, rows in splits.items()}, "images per split")

## Train (or reload) the selected models

The training loop is the script's `train_decoder`: AdamW, mean squared error over (time, site), and the checkpoint with the best validation R2 is restored at the end. Ridge refits in seconds and is never cached.

In [ ]:
# Settings that change neither the fitted model nor its scores.
NON_FIT_FIELDS = {"print_every", "eval_batch_size", "models", "datasets", "seeds",
                  "output_dir", "env", "device", "smoke_test"}
ATTENTION_CLASSES = {
    "baseline": BaselineModel,
    "site_token": SiteTokenBaselineModel,
    "site_token_single": SiteTokenBaselineModel,
    "site_token_matched": SiteTokenBaselineModel,
}


'''
fit_settings
The settings that define one model's fit, used to decide whether a cached
run can be reused.

INPUT:
    - model_cfg: comparison.Cfg -> decoder settings of this model

OUTPUT:
    - settings: dict -> fit-relevant fields plus dataset and seed
'''
def fit_settings(model_cfg):
    settings = {key: value for key, value in asdict(model_cfg).items()
                if key not in NON_FIT_FIELDS}
    return {**settings, "dataset": nb_cfg.dataset, "seed": nb_cfg.seed}
# EOF


'''
train_or_load
Fit one model, or reload it when a cached run has identical settings.

INPUT:
    - name: str -> model key (see the table at the top)

OUTPUT:
    - run: dict -> test predictions, curves, best epoch, parameter count
    - model: nn.Module | None -> best-checkpoint attention model (None for ridge)
'''
def train_or_load(name):
    model_cfg = replace(decoder, **model_overrides.get(name, {}))
    if name == "ridge_site":
        alphas = np.logspace(model_cfg.ridge_alpha_min_exp, model_cfg.ridge_alpha_max_exp,
                             model_cfg.ridge_n_alphas)
        return comparison.fit_ridge_site(design, centered, splits, alphas), None
    # end if closed-form ridge
    if name == "ridge_timebin":
        alphas = np.logspace(model_cfg.ridge_alpha_min_exp, model_cfg.ridge_alpha_max_exp,
                             model_cfg.ridge_n_alphas)
        run = comparison.fit_ridge_timebin(
            design, centered, splits, alphas,
            to_tensors(design.reshape(features.shape).astype(np.float32)),
            shapes, model_cfg, nb_cfg.seed, device,
        )
        return run, None
    # end if SGD ridge

    settings = fit_settings(model_cfg)
    run_path, weights_path, settings_path = (
        run_dir / f"{name}.npz", run_dir / f"{name}.pt", run_dir / f"{name}.json"
    )
    model = comparison.build_attention_model(ATTENTION_CLASSES[name], shapes, model_cfg, device)
    is_cached = settings_path.is_file() and json.loads(settings_path.read_text()) == settings
    if is_cached and not nb_cfg.retrain:
        model.load_state_dict(torch.load(weights_path, map_location=device))
        print(f"{name}: reloaded from {run_dir}")
        return dict(np.load(run_path)), model.eval()
    # end if a cached run with the same settings exists

    print(f"{name}: training ({model_cfg.epochs} epochs, batch {model_cfg.batch_size})")
    run, model = comparison.fit_attention_model(
        ATTENTION_CLASSES[name], raw_data, shapes, model_cfg, nb_cfg.seed, device,
        single_site=name == "site_token_single", return_model=True,
    )
    # Updates per epoch, so curves of different batch sizes can be aligned.
    n_items = len(splits["train"]) * (n_sites if name == "site_token_single" else 1)
    run["updates_per_epoch"] = math.ceil(n_items / model_cfg.batch_size)
    np.savez_compressed(run_path, **run)
    torch.save(model.state_dict(), weights_path)
    settings_path.write_text(json.dumps(settings, indent=2))
    if device.type == "mps":
        torch.mps.empty_cache()
    # end if MPS memory should be released between runs
    return run, model.eval()
# EOF


runs, models = {}, {}
for name in nb_cfg.models:
    runs[name], models[name] = train_or_load(name)
# end for model

## Test scores

R2 is pooled over every (bin, site) of the test images. `pred std` is the across-image standard deviation of the predictions, averaged over (bin, site): a value near zero means the model collapsed to predicting the mean, which is what the batch-64 baseline does on three0.

In [ ]:
zero_mean = np.zeros(test_targets.shape[1:])
neural_rdms = prepared_rdm_time_series(test_targets, decoder.drsa_rdm_metric, decoder.drsa_rsa_metric)
window = (time_ms >= nb_cfg.summary_window_ms[0]) & (time_ms < nb_cfg.summary_window_ms[1])

scores = {}
for name, run in runs.items():
    predictions = run["test_predictions"]
    prediction_rdms = prepared_rdm_time_series(
        predictions, decoder.drsa_rdm_metric, decoder.drsa_rsa_metric
    )
    # R2 of every site over the summary window: [sites].
    residual = ((predictions - test_targets)[:, window] ** 2).sum(axis=(0, 1))
    total = (test_targets[:, window] ** 2).sum(axis=(0, 1))
    scores[name] = {
        "r2": comparison.overall_r2(predictions, test_targets, zero_mean),
        "r2_by_bin": comparison.r2_by_timebin(predictions, test_targets, zero_mean),
        "corr_by_bin": np.nanmean(stimulus_correlation(predictions, test_targets), axis=1),
        "rsa_by_bin": (neural_rdms * prediction_rdms).sum(axis=1),
        "r2_by_site": 1.0 - residual / total,
        "pred_std": float(predictions.std(axis=0).mean()),
    }
# end for model

print(f"{'model':<20}{'params':>11}{'test R2':>9}{'corr':>8}{'RSA':>8}{'pred std':>10}{'best ep':>9}")
for name, row in scores.items():
    run = runs[name]
    print(f"{name:<20}{int(run['n_parameters']):>11,}{row['r2']:>9.4f}"
          f"{np.nanmean(row['corr_by_bin']):>8.4f}{np.nanmean(row['rsa_by_bin']):>8.4f}"
          f"{row['pred_std']:>10.4f}{int(run.get('best_epoch', -1)):>9}")
# end for model row
print(f"\ntest target std {test_targets.std(axis=0).mean():.4f}")

## Learning curves

Left: train (dashed) and validation (solid) MSE per epoch, with ridge's validation MSE as the dotted line. Middle: validation R2 per epoch. Right: validation R2 against optimizer updates, which is the fair axis when batch sizes differ.

In [ ]:
colors, labels = comparison.MODEL_COLORS, comparison.MODEL_LABELS
trained = [name for name in runs if "validation_r2_curve" in runs[name]]
figure, axes = plt.subplots(1, 3, figsize=(19, 4.8))
for name in trained:
    run = runs[name]
    epochs = np.arange(1, len(run["validation_r2_curve"]) + 1)
    axes[0].plot(epochs, run["validation_mse_curve"], "-", lw=2, color=colors[name], label=labels[name])
    axes[0].plot(epochs, run["train_mse_curve"], "--", lw=1.5, color=colors[name])
    axes[1].plot(epochs, run["validation_r2_curve"], lw=2, color=colors[name], label=labels[name])
    best = int(run["best_epoch"])
    if best > 0:
        axes[1].plot(best, run["validation_r2_curve"][best - 1], "D", ms=8, color=colors[name])
    # end if the best checkpoint is a trained epoch
    if "updates_per_epoch" in run:
        axes[2].plot(epochs * int(run["updates_per_epoch"]), run["validation_r2_curve"],
                     lw=2, color=colors[name], label=labels[name])
    # end if the update count was stored
# end for trained model
if "ridge_site" in runs:
    axes[0].axhline(runs["ridge_site"]["validation_mse"], color=colors["ridge_site"], ls=":",
                    lw=2, label=f"{labels['ridge_site']} (validation)")
# end if ridge was fitted
axes[2].set_xscale("log")
for axis, (xlabel, ylabel, title) in zip(axes, (
    ("Epoch", "MSE", "Loss (solid validation, dashed train)"),
    ("Epoch", "Validation R2", "Validation R2 (diamond = kept)"),
    ("Optimizer updates", "Validation R2", "Validation R2 per update"),
)):
    axis.set(xlabel=xlabel, ylabel=ylabel, title=f"{nb_cfg.dataset}: {title}")
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
# end for axis
figure.tight_layout()
plt.show()

## Test metrics per time bin and per site

In [ ]:
figure, axes = plt.subplots(1, 4, figsize=(24, 4.8))
for axis, (key, ylabel) in zip(axes[:3], (
    ("r2_by_bin", "Test R2 (pooled over sites)"),
    ("corr_by_bin", "Mean Pearson r over sites"),
    ("rsa_by_bin", f"{decoder.drsa_rsa_metric} RDM correlation"),
)):
    for name in scores:
        axis.plot(time_ms, scores[name][key], lw=2, color=colors[name], label=labels[name])
    # end for model
    axis.axhline(0, color="gray", lw=0.8)
    axis.set(xlabel="Time from image onset (ms)", ylabel=ylabel)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
# end for per-bin panel

# Per-site R2 against ridge: points above the diagonal beat ridge on that site.
reference = scores["ridge_site"]["r2_by_site"]
for name in scores:
    if name == "ridge_site":
        continue
    # end if this is the reference
    n_better = int((scores[name]["r2_by_site"] > reference).sum())
    axes[3].scatter(reference, scores[name]["r2_by_site"], s=40, color=colors[name],
                    edgecolor="white", label=f"{labels[name]} ({n_better}/{n_sites} above)")
# end for model
limits = [min(axes[3].get_xlim()[0], axes[3].get_ylim()[0]),
          max(axes[3].get_xlim()[1], axes[3].get_ylim()[1])]
axes[3].plot(limits, limits, color="gray", lw=0.8)
axes[3].set(xlabel=f"Ridge R2 per site ({nb_cfg.summary_window_ms[0]:.0f}-{nb_cfg.summary_window_ms[1]:.0f} ms)",
            ylabel="Model R2 per site", title="Per-site test R2 vs ridge")
axes[3].grid(alpha=0.25)
axes[3].legend(fontsize=8)
figure.tight_layout()
plt.show()

## Attention maps

Attention is collected on the test images with the best checkpoint, in eval mode (no dropout), and averaged over heads and images. The baseline returns `[B, T/R, L, E]`; the site-token model returns `[B, T/R, N, L, E]`, one map per (time block, site). Each coarse query covers `temporal_compression_ratio` fine bins, so block $t$ spans $[t R \cdot 10,\, (t+1) R \cdot 10)$ ms.

In [ ]:
'''
collect_attention
Average one attention model's feature attention over the test images.

INPUT:
    - model: nn.Module -> BaselineModel or SiteTokenBaselineModel
    - inputs: torch.Tensor -> [images, layers, embedding] cached features

OUTPUT:
    - mean_attention: np.ndarray -> [T/R, L, E] or [T/R, N, L, E], x uniform
    - layer_mass: np.ndarray -> [images, T/R, L] or [images, T/R, N, L],
      the share of attention each layer receives (sums to 1 over L)
'''
def collect_attention(model, inputs):
    attention_sum, layer_mass = 0.0, []
    model.eval()
    with torch.no_grad():
        for chunk in inputs.split(nb_cfg.attention_batch_size):
            _, attention = model(chunk, use_precomputed_features=True)
            attention_sum = attention_sum + attention.sum(dim=0)
            # aggregate_attention_by_layer expects [B, T, L, E]; fold sites into time.
            folded = attention.reshape(len(chunk), -1, *attention.shape[-2:])
            layer_mass.append(
                aggregate_attention_by_layer(folded).reshape(*attention.shape[:-1]).cpu()
            )
        # end for image chunk
    # end with no gradient tracking
    n_coordinates = attention.shape[-2] * attention.shape[-1]
    mean_attention = (attention_sum / len(inputs)).cpu().numpy() * n_coordinates
    return mean_attention, torch.cat(layer_mass).numpy()
# EOF


test_inputs = raw_data["test"][0]
attention = {name: collect_attention(model, test_inputs)
             for name, model in models.items() if model is not None}
ratio = decoder.temporal_compression_ratio
block_ms = 1000.0 / decoder.target_fs * ratio
block_labels = [f"{t * block_ms:.0f}" for t in range(n_time // ratio)]
for name, (mean_attention, _) in attention.items():
    print(f"{name:<20} mean attention {tuple(mean_attention.shape)} | max {mean_attention.max():.1f}x uniform")
# end for model

### Which layer each time block reads

Share of the attention mass on each DINOv3 layer per time block (1/3 everywhere = no preference), averaged over test images and, for the site-token models, over sites. The error bars on the right are the across-image standard deviation: since the logits are scaled by $\tilde x$, the attention is stimulus-dependent.

In [ ]:
n_models = len(attention)
figure, axes = plt.subplots(2, n_models, figsize=(5.5 * n_models, 7.5), squeeze=False)
for column, (name, (_, layer_mass)) in enumerate(attention.items()):
    # Average over sites first when present: [B, T/R, (N,) L] -> [B, T/R, L].
    per_image = layer_mass.mean(axis=2) if layer_mass.ndim == 4 else layer_mass
    deviation = per_image.mean(axis=0) - 1 / n_layers  # [T/R, L], 0 = uniform
    limit = max(np.abs(deviation).max(), 1e-6)
    image = axes[0, column].imshow(deviation.T, aspect="auto", cmap="RdBu_r", vmin=-limit, vmax=limit)
    axes[0, column].set(xticks=range(len(block_labels)), xticklabels=block_labels,
                        yticks=range(n_layers), yticklabels=[f"layer {label}" for label in layer_labels],
                        xlabel="Block start (ms)", title=labels[name])
    figure.colorbar(image, ax=axes[0, column], label="Layer share - 1/3")

    # Same numbers as lines, with the across-image standard deviation.
    block_centers = (np.arange(per_image.shape[1]) + 0.5) * block_ms
    for layer_index, layer_label in enumerate(layer_labels):
        axes[1, column].errorbar(block_centers, per_image[:, :, layer_index].mean(0),
                                 yerr=per_image[:, :, layer_index].std(0), capsize=2, marker="o",
                                 ms=4, color=f"C{layer_index}", label=f"layer {layer_label}")
    # end for layer
    axes[1, column].axhline(1 / n_layers, color="gray", lw=0.8)
    axes[1, column].set(xlabel="Time (ms)", ylabel="Layer share", title="mean +- image std")
    axes[1, column].grid(alpha=0.25)
    axes[1, column].legend(fontsize=7)
# end for attention model
figure.tight_layout()
plt.show()

### How peaked the feature attention is

Normalized entropy of the mean attention over the 3072 coordinates per time block: 1 is uniform, 0 is one coordinate. The site-token curve is the average over sites, with the min-max range across sites shaded.

In [ ]:
'''
normalized_entropy
Entropy of attention distributions over their last two axes, divided by
log(L * E) so that uniform attention scores 1.

INPUT:
    - attention_map: np.ndarray -> [..., L, E] attention in multiples of uniform

OUTPUT:
    - entropy: np.ndarray -> [...] normalized entropy
'''
def normalized_entropy(attention_map):
    flat = attention_map.reshape(*attention_map.shape[:-2], -1)
    probabilities = flat / flat.sum(axis=-1, keepdims=True)
    entropy = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=-1)
    return entropy / np.log(flat.shape[-1])
# EOF


figure, axis = plt.subplots(figsize=(7, 4))
for name, (mean_attention, _) in attention.items():
    entropy = normalized_entropy(mean_attention)  # [T/R] or [T/R, N]
    block_centers = (np.arange(entropy.shape[0]) + 0.5) * block_ms
    if entropy.ndim == 2:
        axis.fill_between(block_centers, entropy.min(1), entropy.max(1), color=colors[name], alpha=0.15)
        entropy = entropy.mean(1)
    # end if one entropy per site
    axis.plot(block_centers, entropy, "o-", lw=2, color=colors[name], label=labels[name])
# end for attention model
axis.set(xlabel="Time (ms)", ylabel="Normalized entropy", title="Peakedness of the mean feature attention")
axis.grid(alpha=0.25)
axis.legend(fontsize=8)
plt.show()

### Site-token models: layer share per site and time

One panel per layer: rows are sites, columns are time blocks. Differences between rows are what the site tokens buy: with the baseline every site shares one map per time block.

In [ ]:
for name, (_, layer_mass) in attention.items():
    if layer_mass.ndim != 4:
        continue
    # end if the model has no site axis
    mean_mass = layer_mass.mean(axis=0)  # [T/R, N, L]
    figure, axes = plt.subplots(1, n_layers, figsize=(5 * n_layers, 0.18 * n_sites + 2), sharey=True)
    for layer_index, axis in enumerate(axes):
        image = axis.imshow(mean_mass[:, :, layer_index].T, aspect="auto", cmap="viridis",
                            vmin=mean_mass.min(), vmax=mean_mass.max())
        axis.set(xticks=range(len(block_labels)), xticklabels=block_labels,
                 xlabel="Block start (ms)", title=f"{name}: layer {layer_labels[layer_index]}")
    # end for layer
    axes[0].set(yticks=range(n_sites), ylabel="Site")
    figure.colorbar(image, ax=axes, label="Layer share")
    plt.show()
# end for site-token model

### Feature-level maps

Top: the `n_top_features` coordinates with the highest mean attention for each model, across time blocks (site-token maps averaged over sites). Bottom (site-token models): the same coordinates for every site at one chosen block. Values are multiples of uniform on a log color scale.

In [ ]:
from matplotlib.colors import LogNorm

chosen_block = 4  # coarse block index for the per-site panel (4 -> 120-150 ms with R = 3)

for name, (mean_attention, _) in attention.items():
    site_mean = mean_attention.mean(axis=1) if mean_attention.ndim == 4 else mean_attention  # [T/R, L, E]
    flat = site_mean.reshape(site_mean.shape[0], -1)  # [T/R, L * E]
    top = np.argsort(flat.mean(axis=0))[::-1][: nb_cfg.n_top_features]
    top_labels = [f"L{layer_labels[index // feature_dim]}:{index % feature_dim}" for index in top]
    has_sites = mean_attention.ndim == 4
    figure, axes = plt.subplots(1 + has_sites, 1, figsize=(14, 3.5 + 0.18 * n_sites * has_sites))
    axes = np.atleast_1d(axes)
    image = axes[0].imshow(flat[:, top].T, aspect="auto", cmap="magma", norm=LogNorm())
    axes[0].set(xticks=range(len(block_labels)), xticklabels=block_labels, xlabel="Block start (ms)",
                yticks=range(len(top)), yticklabels=top_labels,
                title=f"{labels[name]}: top {len(top)} coordinates over time")
    axes[0].tick_params(axis="y", labelsize=6)
    figure.colorbar(image, ax=axes[0], label="x uniform")
    if has_sites:
        per_site = mean_attention[chosen_block].reshape(n_sites, -1)[:, top]  # [N, top]
        image = axes[1].imshow(per_site, aspect="auto", cmap="magma", norm=LogNorm())
        axes[1].set(xticks=range(len(top)), xticklabels=top_labels, ylabel="Site",
                    title=f"Per site at block {block_labels[chosen_block]} ms")
        axes[1].tick_params(axis="x", labelsize=6, rotation=90)
        figure.colorbar(image, ax=axes[1], label="x uniform")
    # end if the model has a site axis
    figure.tight_layout()
    plt.show()
# end for attention model

### Do sites with similar responses attend to similar features?

Left: correlation between sites' mean attention maps (flattened over time and coordinates). Right: correlation between the same sites' tuning, i.e. their mean test response in the summary window across images (per-bin responses are too noisy on three0 to correlate). Sites are ordered by hierarchical clustering of the response correlation, the same order in both panels. If the site tokens learned something about the sites, the two matrices should share block structure; the number in the title is the correlation of their off-diagonal entries.

In [ ]:
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform

# Tuning per site: mean response in the summary window, [sites, test images].
response_corr = np.corrcoef(test_targets[:, window].mean(axis=1).T)
order = leaves_list(linkage(squareform(1 - response_corr, checks=False), "average"))
off_diagonal = ~np.eye(n_sites, dtype=bool)

for name, (mean_attention, _) in attention.items():
    if mean_attention.ndim != 4:
        continue
    # end if the model has no site axis
    # [T/R, N, L, E] -> [N, T/R * L * E], log so the few peaked coordinates do not dominate.
    site_maps = np.log(mean_attention.transpose(1, 0, 2, 3).reshape(n_sites, -1))
    attention_corr = np.corrcoef(site_maps)
    agreement = np.corrcoef(attention_corr[off_diagonal], response_corr[off_diagonal])[0, 1]
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    for axis, (matrix, title) in zip(axes, (
        (attention_corr, f"{name}: attention-map correlation"),
        (response_corr, "Tuning correlation (test)"),
    )):
        # Symmetric color limits from the off-diagonal entries of each matrix.
        limit = np.abs(matrix[off_diagonal]).max()
        image = axis.imshow(matrix[np.ix_(order, order)], cmap="RdBu_r", vmin=-limit, vmax=limit)
        axis.set(xticks=range(n_sites), xticklabels=order, yticks=range(n_sites),
                 yticklabels=order, title=title)
        axis.tick_params(labelsize=6)
        figure.colorbar(image, ax=axis)
    # end for matrix
    figure.suptitle(f"off-diagonal agreement r = {agreement:.2f}")
    figure.tight_layout()
    plt.show()
# end for site-token model

## Example time courses

Observed (black) and predicted test responses for one site, on the test images with the strongest and weakest mean response in the summary window.

In [ ]:
example_site = int(np.argmax(scores["ridge_site"]["r2_by_site"]))  # best-predicted site; change freely
response = test_targets[:, window, example_site].mean(axis=1)
example_images = [*np.argsort(response)[::-1][:3], *np.argsort(response)[:2]]

figure, axes = plt.subplots(1, len(example_images), figsize=(4 * len(example_images), 3.4), sharey=True)
for axis, image_index in zip(axes, example_images):
    axis.plot(time_ms, test_targets[image_index, :, example_site], color="black", lw=2.5, label="observed")
    for name, run in runs.items():
        axis.plot(time_ms, run["test_predictions"][image_index, :, example_site],
                  color=colors[name], lw=1.8, label=labels[name])
    # end for model
    axis.set(xlabel="Time (ms)", title=f"test image {image_index}")
    axis.grid(alpha=0.25)
# end for example image
axes[0].set_ylabel(f"Site {example_site} (centered)")
axes[0].legend(fontsize=7)
figure.tight_layout()
plt.show()